# ML pipeline (PySpark) for Credit Risk Prediction

This notebook uses the **small dataset** for development and testing. Once validated, the same pipeline should be executed on a big data cluster (e.g., GCP Dataproc) — first validating the cluster with the small dataset and then running on the full big dataset.

In [1]:
# Cell 1: Initialize Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditRiskSmallData") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(spark.version)

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Cell 2: Load small dataset
small_csv = r"/mnt/data/small_dataset.csv"
df = spark.read.option("header", "true").option("inferSchema","true").csv(small_csv)
df.printSchema()
df.show(5)

In [ ]:
# Cell 3: Basic EDA - counts, missing values, class balance
from pyspark.sql.functions import col, isnan, when, count

print('Total rows:', df.count())
df.groupBy('Loan_Default_Status').count().show()
missing_counts = df.select([count(when(col(c).isNull() | isnan(c) | (col(c) == ''), c)).alias(c) for c in df.columns])
missing_counts.show(truncate=False)

In [ ]:
# Cell 4: Data cleaning - imputation and type casting
from pyspark.ml.feature import Imputer
from pyspark.sql.functions import when, col
for c in ['Credit_Score','Annual_Income','Existing_Debt','Loan_Amount','Debt_to_Income_Ratio','Transaction_Behavior_Score']:
    df = df.withColumn(c, col(c).cast('double'))
imputer = Imputer(inputCols=['Credit_Score','Annual_Income','Existing_Debt','Loan_Amount','Debt_to_Income_Ratio','Transaction_Behavior_Score'], outputCols=['Credit_Score_imp','Annual_Income_imp','Existing_Debt_imp','Loan_Amount_imp','Debt_to_Income_Ratio_imp','Transaction_Behavior_Score_imp'])
df = imputer.fit(df).transform(df)
df = df.fillna({'Employment_Status':'Unknown', 'Gender':'Unknown', 'Loan_Purpose':'Unknown'})
for orig, imp in [('Credit_Score','Credit_Score_imp'),('Annual_Income','Annual_Income_imp'),('Existing_Debt','Existing_Debt_imp'),('Loan_Amount','Loan_Amount_imp'),('Debt_to_Income_Ratio','Debt_to_Income_Ratio_imp'),('Transaction_Behavior_Score','Transaction_Behavior_Score_imp')]:
    df = df.drop(orig).withColumnRenamed(imp, orig)
df.printSchema()

In [ ]:
# Cell 5: Feature engineering and selection
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
cat_cols = ['Gender','Employment_Status','Loan_Purpose']
indexers = [StringIndexer(inputCol=c, outputCol=c + '_idx', handleInvalid='keep') for c in cat_cols]
pipeline = Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)
feature_cols = ['Customer_Age','Annual_Income','Credit_Score','Existing_Debt','Loan_Amount','Loan_Term_Months','Debt_to_Income_Ratio','Transaction_Behavior_Score'] + [c + '_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
df = assembler.transform(df)
from pyspark.sql.functions import col
df = df.withColumn('label', col('Loan_Default_Status').cast('double'))
df.select('features','label').show(3, truncate=False)

In [ ]:
# Cell 6: Train/validation split
train_df, val_df = df.randomSplit([0.8, 0.2], seed=42)
print('Train:', train_df.count(), 'Validation:', val_df.count())

In [ ]:
# Cell 7: Train Logistic Regression (baseline)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=20)
pipe_lr = Pipeline(stages=[lr])
model_lr = pipe_lr.fit(train_df)
pred_lr = model_lr.transform(val_df)
pred_lr.select('label','probability','prediction').show(5)

In [ ]:
# Cell 8: Train Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=100)
pipe_rf = Pipeline(stages=[rf])
model_rf = pipe_rf.fit(train_df)
pred_rf = model_rf.transform(val_df)
pred_rf.select('label','probability','prediction').show(5)

In [ ]:
# Cell 9: Evaluation - accuracy, precision, recall, F1, ROC-AUC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

def compute_metrics(pred_df):
    preds_and_labels = pred_df.select('prediction','label').rdd.map(lambda r: (float(r['prediction']), float(r['label'])))
    metrics = MulticlassMetrics(preds_and_labels)
    acc = metrics.accuracy
    precision = metrics.precision(1.0)
    recall = metrics.recall(1.0)
    f1 = metrics.fMeasure(1.0)
    evaluator = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction', labelCol='label', metricName='areaUnderROC')
    roc_auc = evaluator.evaluate(pred_df)
    return {'accuracy': acc, 'precision_pos': precision, 'recall_pos': recall, 'f1_pos': f1, 'roc_auc': roc_auc}

print('Logistic Regression metrics:')
print(compute_metrics(pred_lr))
print('Random Forest metrics:')
print(compute_metrics(pred_rf))

In [ ]:
# Cell 10: Save model (example) and next steps
# model_lr.write().overwrite().save('/path/to/save/lr_model')
# model_rf.write().overwrite().save('/path/to/save/rf_model')

# NOTE: After validating the pipeline on the small dataset, deploy to a big data cluster (e.g., GCP Dataproc):
# 1. Upload the same notebook or job to the cluster and validate the cluster setup using the small dataset.
# 2. Run the identical pipeline on the full big dataset (big_dataset.csv) to train at scale.

print('Done.')